# Laboratorio 11 — El disco lleno

**Duración: 45 minutos.**

En el laboratorio 08 aprendiste a ordenar la libreta. Hoy vas a ver **cuánto cuesta no
hacerlo**, con el número del disco en pantalla.

La pregunta de hoy la hace siempre el que paga el almacenamiento:

> Si la tabla tiene las mismas treinta mil filas que ayer, **¿por qué ocupa el doble?**

---

### En el almacén

Cada corrección que hiciste en la libreta no borró la página anterior: escribió una nueva.
Las viejas siguen en el cajón, ocupando lugar, y nadie las va a botar hasta que alguien lo
diga.

Eso no es un defecto. Es lo que te permitió, en el laboratorio 01, leer cómo estaba la
tabla el martes pasado. Pero tiene un precio, y hoy lo vas a medir.


## Paso 0 — Un año de documentos

Primero, a tu espacio de trabajo.

In [1]:
%%sql
-- Celda 0.1
USE mi_espacio

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/09/24 21:16:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Partimos de cero.

In [2]:
%%sql
-- Celda 0.2
DROP TABLE IF EXISTS documentos_lab11

Un año completo de documentos, los mismos treinta mil del laboratorio 04.

In [3]:
%%sql
-- Celda 0.3
CREATE TABLE documentos_lab11 USING iceberg AS
SELECT * FROM curso.dte_2024

## Paso 1 — Medir el disco

Ahora la primera medición, y hay **dos formas de preguntar** que no dan lo mismo.

La primera se la preguntas a la libreta: cuántos archivos de datos cuentan hoy y cuánto
pesan.

In [4]:
%%sql
-- Celda 1.1
SELECT count(*)                AS archivos_vigentes,
       sum(file_size_in_bytes) AS bytes_vigentes
FROM mi_espacio.documentos_lab11.files

archivos_vigentes,bytes_vigentes
1,776514


La segunda se la preguntas al **disco**, que no sabe nada de libretas y solo ve
archivos. Esta celda no lleva `%%sql`: es una orden del sistema, y por eso empieza con
`!`.

> **Esta celda viene escrita, y hay otras dos iguales más adelante.** Una orden del
> sistema no se puede dejar a medias como una sentencia SQL, así que las tres llegan
> completas y se ejecutan tal cual.
>
> Son las tres mediciones del disco, que es lo que venimos a ver hoy. Córrelas en orden y
> anota los números, porque la gracia está en compararlos.

In [5]:
# Celda 1.2
!hdfs dfs -du -s /warehouse/iceberg/mi_espacio.db/documentos_lab11

1633276  4899828  /warehouse/iceberg/mi_espacio.db/documentos_lab11


**Anota los dos números.** No son iguales, y la diferencia tiene explicación: la
libreta cuenta solo los archivos de datos, y el disco cuenta además los manifiestos y los
`metadata.json`, que también ocupan.

Los vas a comparar tres veces hoy.

## Paso 2 — Cinco correcciones

Ahora cinco correcciones seguidas sobre la misma columna. Son cinco `UPDATE` que no
agregan ni quitan una sola fila: solo cambian el estado de algunos documentos.

In [6]:
%%sql
-- Celda 2.1
UPDATE documentos_lab11 SET estado_sii = 'REVISADO'
WHERE tipo_dte = 33 AND fecha_emision BETWEEN DATE '2024-01-01' AND DATE '2024-01-31'

La segunda, sobre febrero.

In [7]:
%%sql
-- Celda 2.2
UPDATE documentos_lab11 SET estado_sii = 'REVISADO'
WHERE tipo_dte = 33 AND fecha_emision BETWEEN DATE '2024-02-01' AND DATE '2024-02-29'

La tercera, sobre marzo.

In [8]:
%%sql
-- Celda 2.3
UPDATE documentos_lab11 SET estado_sii = 'REVISADO'
WHERE tipo_dte = 33 AND fecha_emision BETWEEN DATE '2024-03-01' AND DATE '2024-03-31'

La cuarta, sobre abril.

In [9]:
%%sql
-- Celda 2.4
UPDATE documentos_lab11 SET estado_sii = 'REVISADO'
WHERE tipo_dte = 33 AND fecha_emision BETWEEN DATE '2024-04-01' AND DATE '2024-04-30'

Y la quinta, sobre mayo.

In [10]:
%%sql
-- Celda 2.5
UPDATE documentos_lab11 SET estado_sii = 'REVISADO'
WHERE tipo_dte = 33 AND fecha_emision BETWEEN DATE '2024-05-01' AND DATE '2024-05-31'

Cinco correcciones hechas. Comprueba primero que la tabla tiene las mismas filas que
antes.

In [11]:
%%sql
-- Celda 2.6
SELECT count(*) AS documentos FROM documentos_lab11

documentos
30000


Los mismos treinta mil. Ahora vuelve a medir el disco.

In [12]:
# Celda 2.7
!hdfs dfs -du -s /warehouse/iceberg/mi_espacio.db/documentos_lab11

5647589  16942767  /warehouse/iceberg/mi_espacio.db/documentos_lab11


## Paso 3 — Dónde se fue el espacio

**Creció, y bastante, con las mismas filas.** Ahí está la pregunta del laboratorio.

Para entender dónde se fue, mira la libreta. Seis páginas: la carga y las cinco
correcciones.

In [13]:
%%sql
-- Celda 3.1
SELECT snapshot_id, committed_at, operation
FROM mi_espacio.documentos_lab11.snapshots
ORDER BY committed_at

snapshot_id,committed_at,operation
4603678120508884165,2026-09-24 21:16:19.147,append
7657024626425900625,2026-09-24 21:16:22.984,overwrite
3342317801108304972,2026-09-24 21:16:25.141,overwrite
7126393374523963647,2026-09-24 21:16:26.319,overwrite
1878350924315441673,2026-09-24 21:16:27.723,overwrite
3448344719147503260,2026-09-24 21:16:28.443,overwrite


Y ahora la comparación que lo explica todo. `.files` son los archivos que **cuentan
hoy**; `.all_files` son **todos los que hay**, incluidos los de las páginas viejas que ya
nadie consulta pero que siguen en el disco.

In [14]:
%%sql
-- Celda 3.2
SELECT
    (SELECT count(*)                FROM mi_espacio.documentos_lab11.files)     AS archivos_vigentes,
    (SELECT sum(file_size_in_bytes) FROM mi_espacio.documentos_lab11.files)     AS bytes_vigentes,
    (SELECT count(*)                FROM mi_espacio.documentos_lab11.all_files) AS archivos_todos,
    (SELECT sum(file_size_in_bytes) FROM mi_espacio.documentos_lab11.all_files) AS bytes_todos

archivos_vigentes,bytes_vigentes,archivos_todos,bytes_todos
1,778984,6,4668576


## Paso 4 — Botar las páginas viejas

**Ahí está el espacio.** La diferencia entre las dos columnas es lo que ocupan las
páginas que ya nadie puede consultar sin pedirlas por su identificador.

Ahora las botamos, con el `expire_snapshots` del laboratorio 08. `retain_last => 1` deja
solo la página vigente.

**Ojo, esto no se deshace.** Después de esta celda, el `VERSION AS OF` de las páginas
viejas deja de funcionar. En producción, la fecha de corte es una decisión que se toma
antes, no después.

> **La fecha de esta celda es de laboratorio y no se copia a producción.** Dice 2030, una
> fecha que todavía no llega, y para el procedimiento eso significa *toda la historia es
> vieja*. Está puesta así para que hoy veas el efecto completo en una sola celda.
>
> En producción esa fecha es una decisión de retención que se toma **antes** de escribir la
> sentencia, y es la de la última compactación, como en el laboratorio 08. **Una fecha
> futura ahí se lleva toda la historia de la tabla**, y con ella cualquier viaje al pasado
> que alguien necesite mañana.

In [15]:
%%sql
-- Celda 4.1
CALL spark_catalog.system.expire_snapshots(
    table       => 'mi_espacio.documentos_lab11',
    older_than  => TIMESTAMP '2030-01-01 00:00:00',
    retain_last => 1
)

deleted_data_files_count,deleted_position_delete_files_count,deleted_equality_delete_files_count,deleted_manifest_files_count,deleted_manifest_lists_count,deleted_statistics_files_count
5,0,0,9,5,0


Vuelve a medir, por tercera y última vez.

In [16]:
# Celda 4.2
!hdfs dfs -du -s /warehouse/iceberg/mi_espacio.db/documentos_lab11

1684910  5054730  /warehouse/iceberg/mi_espacio.db/documentos_lab11


**Bajó.** Y comprueba que los datos vigentes no se tocaron.

In [17]:
%%sql
-- Celda 4.3
SELECT count(*) AS documentos FROM documentos_lab11

documentos
30000


## Paso 5 — Los archivos que nadie reclama

Treinta mil, los mismos de siempre. Se botó la historia, no los datos.

Queda una última herramienta, y es distinta de la anterior. `expire_snapshots` borra lo
que la libreta declara como viejo. `remove_orphan_files` borra lo que **la libreta no
declara en absoluto**: archivos que quedaron en el directorio porque una escritura se
cortó a la mitad.

Y se corre como la viste en el laboratorio 08: con **`dry_run => true`**, que no borra
nada y solo lista lo que borraría. Se mira antes de borrar porque este procedimiento se
lleva **todo archivo que la tabla no reclame**, y un archivo que un trabajo está
escribiendo en este momento todavía no está declarado en ninguna página: para el
procedimiento es un huérfano. Primero se mira; borrar es el segundo paso, y solo con la
lista a la vista.

In [18]:
%%sql
-- Celda 5.1
CALL spark_catalog.system.remove_orphan_files(
    table      => 'mi_espacio.documentos_lab11',
    older_than => TIMESTAMP '2026-09-01 00:00:00',
    dry_run    => true
)

orphan_file_location


## Cierre

Los tres números que anotaste cuentan la historia completa:

| Momento | Qué pasó |
|---|---|
| Después de cargar | el tamaño de partida |
| Después de cinco correcciones | creció, con las mismas treinta mil filas |
| Después de expirar | volvió a bajar |

**La libreta guarda todo hasta que alguien le dice qué botar.** Eso no es un default que
haya que corregir: es la propiedad que te dejó viajar al pasado en el laboratorio 01.

Lo que sí es una decisión, y de las importantes, es **cuánta historia se conserva**. Se
llama política de retención, y hay que tomarla con dos preguntas sobre la mesa:

- **¿Cuánto hacia atrás necesita la Contraloría poder preguntar?** Eso fija el mínimo.
- **¿Cuánto disco hay?** Eso fija el máximo.

Si nadie la toma, el default es *guardar todo para siempre*, y el disco se llena un
viernes a las seis de la tarde.
